# MultiScale CenterLoRA-ESM2 v2 — development-only screen

This notebook tests one controlled change to the v1 development champion: learned fusion of the contextual central lysine with radius-2 and radius-5 contextual mean pools. It trains seed 42 for all 15 epochs on the identical protein-grouped development split. It does **not** read the independent test or perform a full-data refit. Select a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Add a fine-grained read-only GitHub token to Colab Secrets as `GITHUB_TOKEN` and enable notebook access.

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
    try:
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
except ImportError:
    GITHUB_TOKEN = None

git_environment = os.environ.copy()
git_environment['GIT_TERMINAL_PROMPT'] = '0'
askpass_path = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass_path.chmod(0o700)
    git_environment['GIT_ASKPASS'] = str(askpass_path)
    git_environment['GITHUB_TOKEN'] = GITHUB_TOKEN

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--recurse-submodules',
            'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
            str(REPO_ROOT),
        ], check=True, env=git_environment)
    else:
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'],
            check=True, env=git_environment,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
            check=True, env=git_environment,
        )
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Git access failed. Add GITHUB_TOKEN to Colab Secrets for a private repository.'
    ) from error
finally:
    askpass_path.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
print('Project:', REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip uninstall -y torchao

In [ ]:
import json
import platform
import sys
import peft
import sklearn
import torch
import transformers

print('Python:', sys.version)
print('Platform:', platform.platform())
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('PEFT:', peft.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Select a GPU runtime before starting the screen.')

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError('Could not open subprocess output stream')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## Preflight tests

In [ ]:
run_live([
    sys.executable, '-u', '-m', 'unittest', 'discover',
    '-s', 'experiment/tests',
])

## Run the predeclared seed-42 screen

The stable Drive path makes the run restart-safe. Completed metrics are reused after a disconnection. An incomplete run restarts from epoch 1. All 15 epochs run, while the best development checkpoint is retained.

In [ ]:
RUN_DIR = Path(
    '/content/drive/MyDrive/MMUbiPred-research/experiments/'
    'center-lora-esm2-multiscale-v2-seed42'
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RUN_DIR / 'validation_metrics.json'

if METRICS_PATH.exists():
    print('Completed metrics already exist; training was skipped:', METRICS_PATH)
else:
    run_live([
        sys.executable, '-u', 'experiment/scripts/train.py',
        '--config', 'experiment/configs/center_lora_esm2_multiscale_v2.json',
        '--seed', '42',
        '--output-dir', RUN_DIR,
        '--development-only',
    ])
print('Persistent run directory:', RUN_DIR)

## Compare directly with the frozen full-15 v1 champion

The code first checks the standard v1 benchmark path and otherwise searches Drive for a valid seed-42 v1 run that completed all 15 epochs. It rejects a comparison if the validation split hashes differ.

In [ ]:
V1_METRICS_PATH = Path(
    '/content/drive/MyDrive/MMUbiPred-research/benchmarks/'
    'center-lora-esm2-full15-three-seed-v1/'
    'center_lora_esm2_full15_v1/seed_42/validation_metrics.json'
)

def valid_full15_v1_metrics(path):
    try:
        metrics = json.loads(path.read_text())
        manifest = json.loads((path.parent / 'run_manifest.json').read_text())
        config = manifest['config']
        return (
            int(config['seed']) == 42
            and int(config['epochs']) == 15
            and config['experiment_name'] == 'center_lora_esm2_full15_v1'
            and metrics['development_selection']['epochs_completed'] == 15
            and not manifest['locked_test_accessed']
        )
    except Exception:
        return False

if not valid_full15_v1_metrics(V1_METRICS_PATH):
    archive_root = Path('/content/drive/MyDrive/MMUbiPred-research')
    matches = [
        path for path in archive_root.rglob('validation_metrics.json')
        if path != METRICS_PATH and valid_full15_v1_metrics(path)
    ]
    if not matches:
        raise FileNotFoundError(
            'No complete full-15 seed-42 v1 result was found on Drive.'
        )
    V1_METRICS_PATH = max(matches, key=lambda path: path.stat().st_mtime)

v1_metrics = json.loads(V1_METRICS_PATH.read_text())
v2_metrics = json.loads(METRICS_PATH.read_text())
v1_manifest = json.loads((V1_METRICS_PATH.parent / 'run_manifest.json').read_text())
v2_manifest = json.loads((METRICS_PATH.parent / 'run_manifest.json').read_text())

v1_hash = v1_manifest['development_split']['validation_indices_sha256']
v2_hash = v2_manifest['development_split']['validation_indices_sha256']
if v1_hash != v2_hash:
    raise RuntimeError(f'Validation split mismatch: v1={v1_hash}, v2={v2_hash}')

display({'v1_metrics_path': str(V1_METRICS_PATH), 'validation_hash': v1_hash})
display({
    'v2_fixed_threshold': v2_metrics['fixed_threshold'],
    'v2_selected_threshold': v2_metrics['validation_selected_threshold'],
    'v2_development_selection': v2_metrics['development_selection'],
    'v2_branch_names': v2_metrics['branch_names'],
    'v2_mean_component_weights': v2_metrics['mean_component_weights'],
    'v2_prediction_diagnostics': v2_metrics['prediction_diagnostics'],
})

In [ ]:
metric_keys = ['mcc', 'accuracy', 'sensitivity', 'specificity', 'auroc', 'auprc']
comparison = {
    metric: {
        'v1': float(v1_metrics['fixed_threshold'][metric]),
        'v2': float(v2_metrics['fixed_threshold'][metric]),
        'v2_minus_v1': (
            float(v2_metrics['fixed_threshold'][metric])
            - float(v1_metrics['fixed_threshold'][metric])
        ),
    }
    for metric in metric_keys
}
comparison['selected_mcc'] = {
    'v1': float(v1_metrics['validation_selected_threshold']['mcc']),
    'v2': float(v2_metrics['validation_selected_threshold']['mcc']),
    'v2_minus_v1': (
        float(v2_metrics['validation_selected_threshold']['mcc'])
        - float(v1_metrics['validation_selected_threshold']['mcc'])
    ),
}
display(comparison)

mcc_gain = comparison['mcc']['v2_minus_v1']
auroc_gain = comparison['auroc']['v2_minus_v1']
auprc_gain = comparison['auprc']['v2_minus_v1']
collapsed = bool(
    v2_metrics['prediction_diagnostics']['single_class_predictions_at_0_5']
)
ranking_safeguard = auroc_gain >= -0.002 and auprc_gain >= -0.002

if not collapsed and mcc_gain >= 0.005 and ranking_safeguard:
    decision = 'GO: v2 passed the predeclared screen; report results before three-seed confirmation.'
elif not collapsed and mcc_gain > 0 and ranking_safeguard:
    decision = 'BORDERLINE: v2 improved but missed the material-gain target; report before any more runs.'
else:
    decision = 'STOP V2: retain v1 and do not spend time on a three-seed v2 run.'

print(decision)
print('Locked independent test accessed:', False)